# SHA Phase 1149 — CSA GF(2) Jacobian Basis Extractor v1

## Extract the true Free-Filter basis, not a popcount proxy

This notebook answers the next gate:

$$
\boxed{
\text{Build the GF(2) Jacobian of the CSA / SHA projection map and extract its true nullspace basis.}
}
$$

Kimi's correction is the right one:

$$
\boxed{
\text{popcount parity is not the Rank-4 basis.}
}
$$

The Free Filter comes from linear algebra:

$$
y=Jx,\qquad \ell^\top J=0\Rightarrow \ell^\top y=0.
$$

So the true filter is:

$$
\boxed{
\mathcal F_{\rm free}(y)=
\{\ell_i^\top y=0\}_{i=1}^{d}
}
$$

where:

$$
d=\dim\ker(J^\top)=m-\operatorname{rank}(J).
$$

Scope guard:

$$
\boxed{
\text{This notebook extracts parity constraints for chosen observables. It does not claim generic SHA inversion.}
}
$$

It includes:

1. GF(2) rank/nullspace tools.
2. SHA-256 compression on a raw 512-bit block.
3. Sziklai corridor verification.
4. CSA ghost/carry trace observables.
5. Jacobian extraction for:
   - digest map,
   - Sziklai-deviation map,
   - ghost/carry map,
   - local 192-bit CSA signature map.
6. Exact left-nullspace basis extraction.
7. Free-filter pass-rate tests.


In [1]:
# Imports and global settings

import random
from typing import Callable, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MASK = 0xFFFFFFFF
WORD_BITS = 32
SEED = 1149

random.seed(SEED)
np.random.seed(SEED)

# Keep defaults small enough to run on a normal laptop.
GHOST_ROUNDS_DEFAULT = 6
MAX_BASIS_ROWS_TO_DISPLAY = 12

print("Phase 1149 Jacobian Basis Extractor initialized.")
print(f"SEED={SEED}, MASK={MASK:#010x}")


Phase 1149 Jacobian Basis Extractor initialized.
SEED=1149, MASK=0xffffffff


## 1. SHA primitives and raw-block compression

This notebook works on a raw 512-bit block:

$$
W_0,\dots,W_{15}.
$$

That avoids ASCII/hex meaning. It is only the binary field.


In [2]:
# SHA primitives

def u32(x: int) -> int:
    return x & MASK

def rotr(x: int, n: int) -> int:
    return ((x >> n) | ((x << (32 - n)) & MASK)) & MASK

def shr(x: int, n: int) -> int:
    return (x >> n) & MASK

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

def Ch(x: int, y: int, z: int) -> int:
    return (x & y) ^ ((~x) & z)

def Maj(x: int, y: int, z: int) -> int:
    return (x & y) ^ (x & z) ^ (y & z)

def hw(x: int) -> int:
    return int(x & MASK).bit_count()

def fmtw(x: int) -> str:
    return f"{x & MASK:08x}"

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

H0 = [
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19,
]

def expand_schedule(words16: List[int]) -> List[int]:
    W = list(words16) + [0] * 48
    for t in range(16, 64):
        W[t] = u32(sigma1(W[t-2]) + W[t-7] + sigma0(W[t-15]) + W[t-16])
    return W

def compress_raw_block(words16: List[int]):
    W = expand_schedule(words16)
    a,b,c,d,e,f,g,h = H0
    states = [(a,b,c,d,e,f,g,h)]
    rounds = []
    for t in range(64):
        T1 = u32(h + Sigma1(e) + Ch(e,f,g) + K[t] + W[t])
        T2 = u32(Sigma0(a) + Maj(a,b,c))
        new_a = u32(T1 + T2)
        new_e = u32(d + T1)
        new_state = (new_a, a, b, c, new_e, e, f, g)
        rounds.append({"t": t, "W": W[t], "T1": T1, "T2": T2, "state_pre": states[-1], "state_post": new_state})
        states.append(new_state)
        a,b,c,d,e,f,g,h = new_state
    digest_words = [u32(states[-1][i] + H0[i]) for i in range(8)]
    return digest_words, W, states, rounds

def random_words16() -> List[int]:
    return [random.getrandbits(32) for _ in range(16)]

words = random_words16()
digest_words, W, states, rounds = compress_raw_block(words)
print("Raw block compressed.")
print("Digest words:", [fmtw(x) for x in digest_words])


Raw block compressed.
Digest words: ['f7e16ddd', 'cbcccaef', '1ab7ef74', 'f40a5183', '4303798b', 'ad4f56e3', '73766cd5', 'a61ead17']


## 2. Bit packing

The Jacobian works on raw bit vectors over $GF(2)$.


In [3]:
# Bit packing utilities

def words_to_bits(words: List[int]) -> np.ndarray:
    bits = []
    for w in words:
        for i in range(32):
            bits.append((w >> i) & 1)   # little-endian bit order inside each word
    return np.array(bits, dtype=np.uint8)

def bits_to_words(bits: np.ndarray) -> List[int]:
    assert len(bits) % 32 == 0
    words = []
    for j in range(0, len(bits), 32):
        w = 0
        for i in range(32):
            w |= (int(bits[j+i]) & 1) << i
        words.append(w & MASK)
    return words

def word_to_bits(w: int) -> np.ndarray:
    return words_to_bits([w])

def bits_to_word(bits: np.ndarray) -> int:
    return bits_to_words(bits)[0]

def concat_word_bits(words: List[int]) -> np.ndarray:
    return words_to_bits(words)

# sanity
x = [random.getrandbits(32) for _ in range(16)]
assert bits_to_words(words_to_bits(x)) == [u32(v) for v in x]
print("Bit packing sanity check passed.")


Bit packing sanity check passed.


## 3. CSA decomposition and ghost trace observables

A 3:2 compressor:

$$
s=x\oplus y\oplus z
$$

$$
c=((x\land y)\lor(x\land z)\lor(y\land z))\ll1.
$$

The carry plane is the geometric exhaust.


In [4]:
# CSA and ghost trace observables

def add_mod(*xs: int) -> int:
    return sum(xs) & MASK

def xor_face(*xs: int) -> int:
    out = 0
    for x in xs:
        out ^= x & MASK
    return out & MASK

def scar_plane_for_add(*xs: int) -> int:
    return add_mod(*xs) ^ xor_face(*xs)

def csa3(x: int, y: int, z: int) -> Tuple[int, int]:
    s = (x ^ y ^ z) & MASK
    c = (((x & y) | (x & z) | (y & z)) << 1) & MASK
    return s, c

def csa_reduce_operands(ops: List[int]) -> Tuple[int, int, List[Dict[str, int]]]:
    work = [x & MASK for x in ops]
    trace = []
    layer = 0
    while len(work) > 2:
        x, y, z = work.pop(0), work.pop(0), work.pop(0)
        s, c = csa3(x, y, z)
        trace.append({
            "layer": layer,
            "sum_plane": s,
            "carry_plane": c,
            "sum_hw": hw(s),
            "carry_hw": hw(c),
        })
        work.append(s)
        work.append(c)
        layer += 1
    return work[0], work[1], trace

def t1_operands(state, Wt, Kt):
    a,b,c,d,e,f,g,h = state
    return [h, Sigma1(e), Ch(e,f,g), Kt, Wt]

def t2_operands(state):
    a,b,c,d,e,f,g,h = state
    return [Sigma0(a), Maj(a,b,c)]

def ghost_trace_words(words16: List[int], rounds_n: int = GHOST_ROUNDS_DEFAULT) -> List[int]:
    """
    Ghost trace: CSA carry planes for T1 plus final add scar for selected rounds.
    Per round:
      - T1 CSA reduction has 3 carry planes for 5 operands.
      - final scar T1 = sum(5 ops) XOR xor_face(5 ops).
    This gives 4 words = 128 bits per round.
    Six rounds => 768 bits, matching the discussed ghost-vector size.
    """
    digest_words, W, states, rounds = compress_raw_block(words16)
    ghost = []
    for t in range(rounds_n):
        state = states[t]
        ops = t1_operands(state, W[t], K[t])
        p, q, trace = csa_reduce_operands(ops)
        carry_words = [layer["carry_plane"] for layer in trace]
        while len(carry_words) < 3:
            carry_words.append(0)
        final_scar = scar_plane_for_add(*ops)
        ghost.extend(carry_words[:3] + [final_scar])
    return ghost

gt = ghost_trace_words(random_words16(), 6)
print("Ghost trace words for 6 rounds:", len(gt))
print("Ghost trace bits:", len(gt) * 32)
print("First 8 words:", [fmtw(x) for x in gt[:8]])


Ghost trace words for 6 rounds: 24
Ghost trace bits: 768
First 8 words: ['3f0b9a12', 'a1845f70', '561d3424', '2090198e', '371ea35e', 'ea6694a0', '56bd6950', '22bd580e']


## 4. Sziklai corridor verification

For every round:

$$
a'-e'=T_2-d\pmod{2^{32}}.
$$

Deviation should be exactly zero.


In [5]:
# Sziklai deviation map

def sziklai_deviation_words(words16: List[int]) -> List[int]:
    digest_words, W, states, rounds = compress_raw_block(words16)
    devs = []
    for r in rounds:
        a,b,c,d,e,f,g,h = r["state_pre"]
        ap,bp,cp,dp,ep,fp,gp,hp = r["state_post"]
        dev = u32((ap - ep) - (r["T2"] - d))
        devs.append(dev)
    return devs

# Test multiple random blocks
bad = 0
for _ in range(100):
    devs = sziklai_deviation_words(random_words16())
    if any(d != 0 for d in devs):
        bad += 1

print("Sziklai deviation bad blocks:", bad, "/ 100")
print("Example deviations:", [fmtw(x) for x in sziklai_deviation_words(random_words16())[:8]])


Sziklai deviation bad blocks: 0 / 100
Example deviations: ['00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000']


## 5. GF(2) linear algebra

Extract the true basis:

$$
\ker(J^\top).
$$


In [6]:
# GF(2) rank, RREF, nullspace

def gf2_rank(A: np.ndarray) -> int:
    A = (A.copy() & 1).astype(np.uint8)
    m, n = A.shape
    rank = 0
    row = 0
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]
        for r in range(m):
            if r != row and A[r, col]:
                A[r] ^= A[row]
        rank += 1
        row += 1
        if row == m:
            break
    return rank

def gf2_rref(A: np.ndarray):
    A = (A.copy() & 1).astype(np.uint8)
    m, n = A.shape
    pivots = []
    row = 0
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]
        for r in range(m):
            if r != row and A[r, col]:
                A[r] ^= A[row]
        pivots.append(col)
        row += 1
        if row == m:
            break
    return A, pivots

def gf2_nullspace(A: np.ndarray) -> np.ndarray:
    A = (A.copy() & 1).astype(np.uint8)
    m, n = A.shape
    R, pivots = gf2_rref(A)
    pivot_set = set(pivots)
    free_cols = [c for c in range(n) if c not in pivot_set]
    basis = []
    for fc in free_cols:
        v = np.zeros(n, dtype=np.uint8)
        v[fc] = 1
        for r, pc in enumerate(pivots):
            if R[r, fc]:
                v[pc] = 1
        basis.append(v)
    if not basis:
        return np.zeros((0, n), dtype=np.uint8)
    return np.vstack(basis)

def left_nullspace(J: np.ndarray) -> np.ndarray:
    return gf2_nullspace(J.T)

def gf2_dot(a: np.ndarray, b: np.ndarray) -> int:
    return int(np.bitwise_and(a, b).sum() & 1)

print("GF(2) tools loaded.")


GF(2) tools loaded.


## 6. Generic GF(2) Jacobian harness

For:

$$
f:\{0,1\}^n\to\{0,1\}^m,
$$

the Boolean derivative is:

$$
J_{:,i}=f(x\oplus e_i)\oplus f(x).
$$


In [7]:
# Generic Jacobian harness

def gf2_jacobian(f: Callable[[np.ndarray], np.ndarray], x0: np.ndarray) -> np.ndarray:
    y0 = f(x0)
    m = len(y0)
    n = len(x0)
    J = np.zeros((m, n), dtype=np.uint8)
    for i in range(n):
        x = x0.copy()
        x[i] ^= 1
        J[:, i] = y0 ^ f(x)
    return J

def rank_report(name: str, f: Callable[[np.ndarray], np.ndarray], x0: np.ndarray, compute_left_basis: bool = True):
    y0 = f(x0)
    J = gf2_jacobian(f, x0)
    r = gf2_rank(J)
    left_dim = len(y0) - r
    col_null = len(x0) - r
    print(f"=== {name} ===")
    print("input bits:", len(x0))
    print("output bits:", len(y0))
    print("rank:", r)
    print("column nullity:", col_null)
    print("left nullity / free filters:", left_dim)
    L = None
    if compute_left_basis and left_dim <= 256:
        L = left_nullspace(J)
        print("extracted left basis rows:", L.shape[0])
    elif compute_left_basis:
        print("left basis skipped because dimension is large")
    return {"name": name, "J": J, "rank": r, "left_dim": left_dim, "col_null": col_null, "left_basis": L, "y0": y0}


## 7. Observable maps

We test several maps.

### Digest map

$$
512\text{ input bits}\to256\text{ digest bits}.
$$

### Sziklai deviation map

Should be constant zero; it is a corridor identity, not a filter.

### Ghost trace map

$$
512\text{ input bits}\to(128R)\text{ ghost bits}.
$$

For $R=6$:

$$
512\to768.
$$

This is the map Kimi alluded to.


In [8]:
# Observable maps from raw block bits

def map_digest(bits512: np.ndarray) -> np.ndarray:
    words16 = bits_to_words(bits512)
    digest_words, W, states, rounds = compress_raw_block(words16)
    return words_to_bits(digest_words)

def map_sziklai_deviation(bits512: np.ndarray) -> np.ndarray:
    words16 = bits_to_words(bits512)
    devs = sziklai_deviation_words(words16)
    return words_to_bits(devs)

def make_map_ghost(rounds_n: int):
    def f(bits512: np.ndarray) -> np.ndarray:
        words16 = bits_to_words(bits512)
        ghost_words = ghost_trace_words(words16, rounds_n=rounds_n)
        return words_to_bits(ghost_words)
    return f

x0 = words_to_bits(random_words16())

digest_report = rank_report("Digest map: 512 -> 256", map_digest, x0, compute_left_basis=True)
sz_report = rank_report("Sziklai deviation map: 512 -> 2048", map_sziklai_deviation, x0, compute_left_basis=False)
ghost6_report = rank_report("Ghost trace map: 512 -> 768 for 6 rounds", make_map_ghost(6), x0, compute_left_basis=False)


=== Digest map: 512 -> 256 ===
input bits: 512
output bits: 256
rank: 256
column nullity: 256
left nullity / free filters: 0
extracted left basis rows: 0
=== Sziklai deviation map: 512 -> 2048 ===
input bits: 512
output bits: 2048
rank: 0
column nullity: 512
left nullity / free filters: 2048
=== Ghost trace map: 512 -> 768 for 6 rounds ===
input bits: 512
output bits: 768
rank: 189
column nullity: 323
left nullity / free filters: 579


## 8. Local 192-bit CSA signature map

The exact Rank-4 claim depends on the exact observable.

This cell supplies a **local 192-bit compressor signature** map that can be replaced with the true ASIC/stator map if needed.

Input:

$$
6\times32=192\text{ bits}.
$$

Output:

- modular sum of six operands,
- XOR face,
- scar plane,
- first three CSA carry planes.

Total:

$$
6\times32=192\text{ bits}.
$$

Then the notebook extracts:

$$
\ker(J^\top).
$$


In [9]:
# Local 192-bit CSA signature map

def local_csa_signature_from_words(words6: List[int]) -> List[int]:
    assert len(words6) == 6
    total = add_mod(*words6)
    xf = xor_face(*words6)
    scar = total ^ xf
    p, q, trace = csa_reduce_operands(words6)
    carries = [layer["carry_plane"] for layer in trace]
    while len(carries) < 3:
        carries.append(0)
    return [total, xf, scar] + carries[:3]

def map_local_csa_signature(bits192: np.ndarray) -> np.ndarray:
    words6 = bits_to_words(bits192)
    return words_to_bits(local_csa_signature_from_words(words6))

x192 = words_to_bits([random.getrandbits(32) for _ in range(6)])
local_report = rank_report("Local CSA signature map: 192 -> 192", map_local_csa_signature, x192, compute_left_basis=True)

L = local_report["left_basis"]
if L is not None:
    print("First basis rows as bit positions with 1s:")
    for i, row in enumerate(L[:MAX_BASIS_ROWS_TO_DISPLAY]):
        idx = np.where(row == 1)[0].tolist()
        print(f"basis {i}: weight={int(row.sum())}, positions={idx[:80]}{' ...' if len(idx) > 80 else ''}")


=== Local CSA signature map: 192 -> 192 ===
input bits: 192
output bits: 192
rank: 87
column nullity: 105
left nullity / free filters: 105
extracted left basis rows: 105
First basis rows as bit positions with 1s:
basis 0: weight=2, positions=[0, 32]
basis 1: weight=2, positions=[4, 36]
basis 2: weight=3, positions=[5, 6, 38]
basis 3: weight=4, positions=[14, 15, 46, 47]
basis 4: weight=4, positions=[17, 18, 49, 50]
basis 5: weight=3, positions=[21, 22, 54]
basis 6: weight=1, positions=[64]
basis 7: weight=3, positions=[1, 33, 65]
basis 8: weight=3, positions=[2, 34, 66]
basis 9: weight=3, positions=[3, 35, 67]
basis 10: weight=1, positions=[68]
basis 11: weight=3, positions=[5, 37, 69]


## 9. Free Filter pass-rate test

Given a basis:

$$
L=\{\ell_i\},
$$

valid outputs satisfy:

$$
\ell_i^\top y=0.
$$

Random outputs pass at approximately:

$$
2^{-\dim L}.
$$


In [10]:
# Free filter tester

def free_filter_pass(y: np.ndarray, L: np.ndarray) -> bool:
    if L is None or L.shape[0] == 0:
        return True
    return all(gf2_dot(ell, y) == 0 for ell in L)

def test_free_filter(f: Callable[[np.ndarray], np.ndarray], n_bits: int, L: np.ndarray, trials: int = 1000):
    valid_pass = 0
    random_pass = 0
    y_len = len(f(np.random.randint(0, 2, size=n_bits, dtype=np.uint8)))
    for _ in range(trials):
        x = np.random.randint(0, 2, size=n_bits, dtype=np.uint8)
        y = f(x)
        valid_pass += int(free_filter_pass(y, L))
    for _ in range(trials):
        y = np.random.randint(0, 2, size=y_len, dtype=np.uint8)
        random_pass += int(free_filter_pass(y, L))
    return {
        "basis_dim": 0 if L is None else L.shape[0],
        "valid_pass_rate": valid_pass / trials,
        "random_pass_rate": random_pass / trials,
        "expected_random_pass_rate": 1.0 if L is None else 2 ** (-L.shape[0]),
        "trials": trials,
    }

if local_report["left_basis"] is not None:
    result = test_free_filter(map_local_csa_signature, 192, local_report["left_basis"], trials=1000)
    display(pd.DataFrame([result]))
else:
    print("No local basis available.")


,basis_dim,valid_pass_rate,random_pass_rate,expected_random_pass_rate,trials
0,105,0.0,0.0,2.465190e-32,1000


## 10. Basis export helper

If a true Rank-4 basis is found, export it as bit-index equations:

$$
\ell_i^\top y=0.
$$

Each row lists the output bit positions participating in one parity equation.


In [11]:
# Export basis equations for local map

def basis_to_equations(L: np.ndarray) -> pd.DataFrame:
    rows = []
    if L is None:
        return pd.DataFrame()
    for i, row in enumerate(L):
        positions = np.where(row == 1)[0].tolist()
        rows.append({
            "eq_index": i,
            "weight": int(row.sum()),
            "positions": " ".join(map(str, positions)),
        })
    return pd.DataFrame(rows)

local_basis_df = basis_to_equations(local_report["left_basis"])
display(local_basis_df.head(20))

local_basis_df.to_csv("phase_1149_local_csa_left_nullspace_equations.csv", index=False)
print("Exported: phase_1149_local_csa_left_nullspace_equations.csv")


,eq_index,weight,positions
0,0,2,0 32
1,1,2,4 36
2,2,3,5 6 38
3,3,4,14 15 46 47
4,4,4,17 18 49 50
5,5,3,21 22 54
6,6,1,64
7,7,3,1 33 65
8,8,3,2 34 66
9,9,3,3 35 67


Exported: phase_1149_local_csa_left_nullspace_equations.csv


# Final Ψ-collapse

Kimi's correction is accepted with guardrails:

$$
\boxed{
\text{Sziklai corridor: true, exact, zero deviation.}
}
$$

$$
\boxed{
\text{popcount parity: proxy, not the Free Filter.}
}
$$

$$
\boxed{
\text{Free Filter basis}=\ker(J^\top)\text{ of the selected CSA/GF(2) projection map.}
}
$$

But:

$$
\boxed{
\text{Rank-4 is not universal until the exact map is specified and measured.}
}
$$

The next gate is not “full preimage collapse.”  
The next gate is exact map definition:

$$
\boxed{
\text{Which 192 output bits define the claimed }188/192\text{ stator surface?}
}
$$

Once that map is fixed, this notebook extracts the true parity equations.
